Import Libraries

In [1]:
from typing import List, Union

import matplotlib as mpl
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from umap import UMAP

W1106 17:45:28.061000 4220 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Settings

In [2]:
# Year the SCOPUS data was retrieved (the dataset's last, partial year). Used for the citation age
# correction and as the last year shown in the time plots.
REFERENCE_YEAR = 2024

# False: load the saved model in ../bertopic/model (the model used in the paper).
# True: fit a new model and save it to ../bertopic/model, overwriting the saved one. UMAP results are not
# reproducible across machines and library versions, so a new fit gives different topics and topic_labels
# (below) must then be rewritten.
FIT_MODEL = False

# Topics highlighted in the "selected" figures
SELECTED_TOPICS = [0, 1, 2, 3, 4, 5, 6, 9, 11, 15, 19, 21, 24, 38]

# Colour palette for topics and basins
COLORS = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#D55E00", "#0072B2", "#CC79A7",
          "#FF0000", "#00FF00", "#0000FF", "#FFFF00", "#00FFFF", "#FF00FF", "#C0C0C0",
          "#808080", "#800000", "#808000", "#008000", "#800080", "#008080"]

Prepare the data

In [3]:
# Prepare the data
data = pd.read_csv('../get_data/alldata.csv')

#Create abstract dataframe (docs) and save
docs = data['Abstract']
docs.to_csv('../bertopic/preprocessed_data/docs.csv')
docs = docs.tolist()

#Create timestamp dataframe and save; This will be used to create topics_over_time
timestamp = data['Year']
timestamp.to_csv('../bertopic/preprocessed_data/timestamp.csv')

# Correct citation count by age of the paper: age = max(REFERENCE_YEAR - Year, 1),
# so papers from the last two years keep their raw count.
cite = pd.DataFrame()
age = (REFERENCE_YEAR - data['Year']).clip(lower=1)
corrected_cite = data['Cited by'] / age

cite['Citation Count'] = data['Cited by']
cite['Citation Count (Corrected by Age)'] = corrected_cite
cite.to_csv('../bertopic/preprocessed_data/cite.csv')

# Create ID dataframe with Title and DOI
ids = data[['Title', 'DOI', 'Authors']]
ids.to_csv('../bertopic/preprocessed_data/id.csv')

# Load class data
classes = pd.read_csv('../NER/merged_fuzzy_classes_labels.csv')
classes = classes['Basin']
classes = classes.fillna('Unknown')

# Load embeddings
embeddings = np.load('../bertopic/preprocessed_data/embeddings.csv.npy')


Define Model

In [4]:
## Define submodels according to the best model chose after tuning hyperparameters
#Dimensionality Reduction
umap_model = UMAP(n_components=11, n_neighbors=8, min_dist=0, metric="cosine", verbose=True, random_state = 42)

#Clustering
hdbscan_model = HDBSCAN(min_cluster_size=47, min_samples=2, metric='euclidean', cluster_selection_epsilon=0.15, cluster_selection_method='eom', prediction_data=True)

#Topic Representations: Vectorizer
vectorizer_model = CountVectorizer(ngram_range=(1, 2), stop_words="english") #this excludes english stop words and ngram_range=(1,2) aims to includes compound words like "plastic waste"

# Set embedding model
sentence_model = SentenceTransformer("allenai-specter")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
if FIT_MODEL:
    # Define topic model using pre-defined sub-models
    topic_model = BERTopic(umap_model=umap_model,
                           hdbscan_model=hdbscan_model,
                           vectorizer_model=vectorizer_model,
                           embedding_model=sentence_model,
                           calculate_probabilities=True,  # Allows a more in-depth look into the probability of each doc belonging to each topic
                           verbose=True)  # Training can take a while
    topics, probs = topic_model.fit_transform(docs, embeddings)
    topic_model.save("../bertopic/model", serialization="safetensors", save_ctfidf=True,
                     save_embedding_model="allenai-specter")
else:
    topic_model = BERTopic.load("../bertopic/model", embedding_model=sentence_model)
    # Representative documents are not stored by the safetensors serialization; recompute them from the
    # saved topic assignments with BERTopic's own (seeded) procedure
    documents = pd.DataFrame({"Document": docs, "ID": range(len(docs)), "Topic": topic_model.topics_, "Image": None})
    topic_model._save_representative_docs(documents)

# Topic ids, excluding the outlier topic (-1)
topic_ids = sorted(topic for topic in topic_model.get_topics() if topic != -1)
print(f"{len(topic_ids)} topics")

# Try freq to see topic names and check how many documents there are in each topic
freq = topic_model.get_topic_info()
freq

39 topics


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2709,-1_microplastics_plastic_marine_microplastic,"[microplastics, plastic, marine, microplastic,...",[With an increasing quantity of plastic waste ...
1,0,1500,0_microplastics_mps_microplastic_water,"[microplastics, mps, microplastic, water, sedi...",[Microplastics (plastics <5 mm) are globally w...
2,1,861,1_exposure_effects_ps_mps,"[exposure, effects, ps, mps, microplastics, np...","[Nowadays, the occurrence of a large volume of..."
3,2,771,2_plastic_waste_marine_pollution,"[plastic, waste, marine, pollution, plastic po...",[Plastic use and production have dramatically ...
4,3,728,3_fish_microplastics_mps_species,"[fish, microplastics, mps, species, microplast...",[Due to its potential impact on food safety an...
5,4,683,4_microbial_plastic_bacterial_plastisphere,"[microbial, plastic, bacterial, plastisphere, ...",[Microplastics have become a widespread concer...
6,5,676,5_litter_debris_marine_marine debris,"[litter, debris, marine, marine debris, items,...",[Marine litter is a complex and multi-dimensio...
7,6,363,6_ocean_plastic_debris_surface,"[ocean, plastic, debris, surface, model, float...",[Although the Indian Ocean receives a large am...
8,7,322,7_soil_mps_soils_microplastics,"[soil, mps, soils, microplastics, groundwater,...",[Microplastic is an emerging contaminant of co...
9,8,318,8_microplastics_mps_ocean_sea,"[microplastics, mps, ocean, sea, microplastic,...",[ Microplastics ar...


Get top words and representative docs

In [6]:
# Top words per topic
pd.DataFrame({topic: [word for word, _ in topic_model.get_topic(topic)] for topic in topic_ids}).T

,0,1,2,3,4,5,6,7,8,9
0,microplastics,mps,microplastic,water,sediment,sediments,samples,river,abundance,particles
1,exposure,effects,ps,mps,microplastics,nps,exposed,toxicity,organisms,marine
2,plastic,waste,marine,pollution,plastic pollution,plastic waste,environmental,management,plastics,marine plastic
3,fish,microplastics,mps,species,microplastic,mp,study,particles,individual,contamination
4,microbial,plastic,bacterial,plastisphere,communities,bacteria,plastics,community,degradation,marine
5,litter,debris,marine,marine debris,items,beach,beaches,marine litter,plastic,waste
6,ocean,plastic,debris,surface,model,floating,marine,plastic debris,drift,sea
7,soil,mps,soils,microplastics,groundwater,agricultural,microplastic,mp,terrestrial,plastic
8,microplastics,mps,ocean,sea,microplastic,surface,abundance,seawater,distribution,water
9,plastic,ingestion,plastic ingestion,species,ingested,seabirds,turtles,debris,plastics,marine


In [7]:
topic_number = 32
print(f"Topic {topic_number}")
print("[KEYWORDS:SCORES]", topic_model.get_topic(topic_number))
print("[DOCUMENTS]", topic_model.get_representative_docs(topic_number))

Topic 32
[KEYWORDS:SCORES] [['separation', 0.02916597884782645], ['extraction', 0.023549912804926305], ['recovery', 0.02083543638550441], ['method', 0.01913681490770246], ['density', 0.017151805459126514], ['microplastics', 0.01541628097440636], ['samples', 0.015116630592622667], ['sediments', 0.01509689745917477], ['sample', 0.014566303943284589], ['sediment', 0.013617113758257035]]
[DOCUMENTS] ['Microplastics are widely distributed in the environment and to define contamination hot spots, environmental samples have to be analysed by means of cost-as well as time-efficient and reliable standardised protocols. Due to the lipophilic characteristics of plastics, oil extraction as a fast and density-independent separation process is beneficial for the crucial extraction step. It was extensively validated (480 experiments) in two test setups by using canola oil and a cost-effective, plastic-free separation unit with spiked microplastics (19 different polymer types) in the density range fro

In [8]:
# Representative documents of each topic, with their DOI and title
paper_info = data.drop_duplicates('Abstract').set_index('Abstract')[['DOI', 'Title']]
rep_docs = pd.DataFrame([(topic, rank, doc)
                         for topic, topic_docs in topic_model.get_representative_docs().items()
                         for rank, doc in enumerate(topic_docs, 1)],
                        columns=['Topic', 'Rank', 'Abstract'])
rep_docs = rep_docs.join(paper_info, on='Abstract')[['Topic', 'Rank', 'DOI', 'Title', 'Abstract']].sort_values(['Topic', 'Rank'])
rep_docs.to_csv('../bertopic/preprocessed_data/rep_docs.csv', index=False)
rep_docs.drop(columns='Abstract')

,Topic,Rank,DOI,Title
0,-1,1,10.1016/j.jenvman.2022.116961,"Global occurrence, drivers, and environmental ..."
1,-1,2,10.1016/j.scitotenv.2024.171992,Microplastics contamination in aquaculture-ric...
2,-1,3,10.1016/j.scitotenv.2023.161704,Characterization of microplastic pollution in ...
3,0,1,10.1016/j.scitotenv.2022.154659,Spatial distribution of microplastics in volca...
4,0,2,10.1007/s11270-021-05297-8,Microplastic Occurrence in the Water and Sedim...
...,...,...,...,...
115,37,2,10.1016/j.envint.2020.105460,Thirty years of marine debris in the Southern ...
116,37,3,10.1016/S0025-326X(99)00234-9,Litter on the sea floor along European coasts
117,38,1,10.1016/j.ecoenv.2020.110356,Organic pollutants in sedimentary microplastic...
118,38,2,10.1016/j.envres.2024.118579,Occurrence of halogenated organic contaminants...


Set Labels

Each topic's full name summarises its top words and its representative documents (see `rep_docs.csv`).
The labels belong to the saved model in `../bertopic/model`; they must be rewritten if the model is refitted.

In [9]:
topic_labels = {
    -1: '-1: Outliers',
    0: '0: Microplastic Abundance in Freshwater and Estuarine Water and Sediments',
    1: '1: Toxic Effects of Micro- and Nanoplastic Exposure on Aquatic Organisms',
    2: '2: Plastic Waste Management, Governance and Mitigation Strategies',
    3: '3: Microplastic Contamination in Tissues of Commercial Fish Species',
    4: '4: Microbial Communities on Plastics (the Plastisphere)',
    5: '5: Beach Litter Surveys and Marine Litter Monitoring',
    6: '6: Modeling the Dispersal of Floating Plastic Debris in the Ocean',
    7: '7: Microplastics in Soils and Terrestrial Ecosystems',
    8: '8: Microplastic Abundance and Distribution in Surface Seawater',
    9: '9: Plastic Ingestion by Marine Fauna: Fish, Seabirds and Turtles',
    10: '10: Reviews of Microplastic Sources, Fate and Effects in Aquatic Environments',
    11: '11: Marine Debris Ingestion and Entanglement in Sea Turtles and Marine Mammals',
    12: '12: Adsorption of Metals and Organic Pollutants onto Microplastics',
    13: '13: Human Exposure to Micro- and Nanoplastics through Food and Health Risks',
    14: '14: Synthetic Microfiber Release from Textile Washing',
    15: '15: Distribution and Composition of Plastic Debris on Beaches',
    16: '16: Settling and Transport Dynamics of Microplastic Particles',
    17: '17: Microplastic Removal in Wastewater Treatment Plants',
    18: '18: Riverine Macroplastic Transport to the Ocean',
    19: '19: Abandoned, Lost or Discarded Fishing Gear and Ghost Fishing',
    20: '20: Deep Learning and Drone Imagery for Marine Debris Detection',
    21: '21: Satellite and Hyperspectral Remote Sensing of Floating Plastics',
    22: '22: Face Mask and PPE Pollution during the COVID-19 Pandemic',
    23: '23: Plastic Additives and Sorbed Organic Contaminants in Marine Plastic Debris',
    24: '24: Toxicity of Micro- and Nanoplastics to Marine Microalgae',
    25: '25: Microplastics in Coral Reef Ecosystems',
    26: '26: Plastic Ingestion across Marine and Freshwater Fish Species',
    27: '27: Atmospheric and Airborne Microplastics',
    28: '28: Global Ocean Plastic Waste Inputs and Climate Impacts',
    29: '29: Aging of Micro- and Nanoplastics and Their Interactions with Contaminants',
    30: '30: Tire and Road Wear Particles in Urban Stormwater Runoff',
    31: '31: Bioaccumulation and Trophic Transfer of Microplastics in Food Webs and Aquaculture',
    32: '32: Separation and Extraction Methods for Microplastic Analysis',
    33: '33: UV Degradation, Weathering and Gasification of Plastics',
    34: '34: Reviews of Microplastic Occurrence and Ingestion by Marine and Freshwater Biota',
    35: '35: Plastic Production, Life Cycle and Recycled Plastics in Construction',
    36: '36: Reviews of Micro- and Nanoplastic Fate and Impacts in Aquatic Ecosystems',
    37: '37: Long-Term Surveys of Beached and Seafloor Marine Debris',
    38: '38: PAHs and Other Organic Pollutants Associated with Microplastics in Sediments'
}
assert set(topic_labels) == {-1, *topic_ids}, "topic_labels does not match the topics of the model"

topic_model.set_topic_labels(topic_labels)

Explore the model

In [10]:
# Calculate topics per class
topics_per_class = topic_model.topics_per_class(docs, classes)

0it [00:00, ?it/s]

1it [00:00,  3.22it/s]

2it [00:00,  3.17it/s]

3it [00:00,  3.03it/s]

4it [00:01,  2.95it/s]

5it [00:01,  2.46it/s]

... (10 outputs left out)

16it [00:06,  2.62it/s]

17it [00:06,  2.66it/s]

18it [00:06,  2.78it/s]

18it [00:06,  2.58it/s]

In [11]:
def visualize_topics_per_class(topic_model,
                               topics_per_class: pd.DataFrame,
                               top_n_topics: int = 10,
                               topics: List[int] = None,
                               classes: List[int] = None,
                               normalize_frequency: bool = False,
                               custom_labels: Union[bool, str] = False,
                               title: str = "<b>Topics per Class</b>",
                               width: int = 1250,
                               height: int = 900) -> go.Figure:
    """ Visualize topics per class """
    
    colors = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#D55E00", "#0072B2", "#CC79A7"]

    # Select topics based on top_n and topics args
    freq_df = topic_model.get_topic_freq()
    freq_df = freq_df.loc[freq_df.Topic != -1, :]
    if topics is not None:
        selected_topics = list(topics)
    elif top_n_topics is not None:
        selected_topics = sorted(freq_df.Topic.to_list()[:top_n_topics])
    else:
        selected_topics = sorted(freq_df.Topic.to_list())
    # Prepare data
    if isinstance(custom_labels, str):
        topic_names = [[[str(topic), None]] + topic_model.topic_aspects_[custom_labels][topic] for topic in topics]
        topic_names = ["_".join([label[0] for label in labels[:4]]) for labels in topic_names]
        topic_names = [label if len(label) < 30 else label[:27] + "..." for label in topic_names]
        topic_names = {key: topic_names[index] for index, key in enumerate(topic_model.topic_labels_.keys())}
    elif topic_model.custom_labels_ is not None and custom_labels:
        topic_names = {key: topic_model.custom_labels_[key + topic_model._outliers] for key, _ in topic_model.topic_labels_.items()}
    else:
        topic_names = {key: value[:40] + "..." if len(value) > 40 else value
                       for key, value in topic_model.topic_labels_.items()}
    topics_per_class["Name"] = topics_per_class.Topic.map(topic_names)
    data = topics_per_class.loc[topics_per_class.Topic.isin(selected_topics), :]

# Z-Score normalization with range 0 to 1
    if normalize_frequency:
        scaler = StandardScaler()
        scaled_values = scaler.fit_transform(data['Frequency'].values.reshape(-1, 1))
        min_value = scaled_values.min()
        max_value = scaled_values.max()
        data['NormalizedFrequency'] = (scaled_values - min_value) / (max_value - min_value)


    # Add traces
    fig = go.Figure()
    for index, topic in enumerate(selected_topics):
        if index == 0:
            visible = True
        else:
            visible = "legendonly"
        trace_data = data.loc[data.Topic == topic, :]
        topic_name = trace_data.Name.values[0]
        words = trace_data.Words.values
        if normalize_frequency:
            x = trace_data.NormalizedFrequency
        else:
            x = trace_data.Frequency
        fig.add_trace(go.Bar(y=trace_data.Class,
                             x=x,
                             visible=visible,
                             marker_color=colors[index % 7],
                             hoverinfo="text",
                             name=topic_name,
                             orientation="h",
                             hovertext=[f'<b>Topic {topic}</b><br>Words: {word}' for word in words]))

    # Styling of the visualization
    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)
    fig.update_layout(
        xaxis_title="Normalized Frequency" if normalize_frequency else "Frequency",
        yaxis_title="Class",
        title={
            'text': f"{title}",
            'y': .95,
            'x': 0.40,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(
                size=22,
                color="Black")
        },
        template="simple_white",
        width=width,
        height=height,
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Rockwell"
        ),
        legend=dict(
            title="<b>Global Topic Representation",
        )
    )
    return fig


# Normalize topics within each class to avoid having the largest topics dominating. Tried z-scaler, x/x.max, x/x.sum and they all return very similar results.

def get_top_topics_per_class(topics_per_class, top_n=5):
    
    # Normalize frequencies within each class
    topics_per_class['NormalizedFrequency'] = topics_per_class.groupby('Class')['Frequency'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    
    # Filter out the outlier topic (-1) and sort by normalized frequency
    filtered_topics = topics_per_class[topics_per_class['Topic'] != -1].sort_values('NormalizedFrequency', ascending=False)
    
    # Select top topics based on normalized frequencies per class
    top_topics_per_class = filtered_topics.groupby('Class').head(top_n)
    
    # Create a dictionary of classes and top topics
    top_topics_dict = top_topics_per_class.groupby('Class')['Topic'].apply(list).to_dict()
    
    return top_topics_dict

top_topics_dict = get_top_topics_per_class(topics_per_class, top_n=5)
top_topics_df = pd.DataFrame.from_dict(top_topics_dict, orient='index')
print(top_topics_df)


unique_topics_list = freq["Topic"].unique()

fig = visualize_topics_per_class(topic_model, topics_per_class, topics=unique_topics_list, custom_labels=True, normalize_frequency=True) # Normalization is made so that smaller topics 
fig

                   0    1    2     3     4
Antarctic          8  0.0  3.0   6.0  11.0
Arctic             8  0.0  9.0   5.0   2.0
Baltic Sea         0  5.0  8.0   4.0   3.0
Black Sea          3  0.0  5.0   4.0   8.0
Equatorial Atlant  0  2.0  5.0   4.0   6.0
Equatorial Indian  0  2.0  3.0   5.0   6.0
Equatorial Pac     5  0.0  3.0   2.0   6.0
Mediteranean       0  5.0  3.0   6.0   1.0
North Atlantic     0  5.0  3.0   2.0   4.0
North Indian       0  3.0  5.0   2.0  10.0
North Pacific      0  3.0  2.0   5.0   4.0
Persian Gulf       0  3.0  5.0   4.0   7.0
Red Sea            0  3.0  1.0  25.0  20.0
South Atlantic     0  5.0  3.0   9.0   8.0
South Indian       0  5.0  2.0   6.0  11.0
South Pacific      5  0.0  9.0  11.0   2.0
Sulu Sea           5  NaN  NaN   NaN   NaN
Unknown            1  4.0  2.0   0.0   7.0


In [12]:
fig = topic_model.visualize_documents(docs, embeddings=embeddings, hide_annotations=True, custom_labels=False)
fig.write_html("../graphs/docs_per_topic.html")
fig

/Users/henriback/code/work/mpl_topics/.venv/lib/python3.12/site-packages/bertopic/plotting/_documents.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selection["text"] = ""
/Users/henriback/code/work/mpl_topics/.venv/lib/python3.12/site-packages/bertopic/plotting/_documents.py:167: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selection.loc[len(selection), :] = [
/Users/henriback/code/work/mpl_topics/.venv/lib/python3.12/site-packages/bertopic/plotting/_documents.py:192: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFr

In [13]:
fig = topic_model.visualize_barchart(top_n_topics=len(topic_ids), n_words = 10, width=400, height=400) # you can change top_n_topics to see more or less topics
fig.write_html("../graphs/word_scores.html")
fig

In [14]:
fig = topic_model.visualize_heatmap(custom_labels=False)
fig.write_html("../graphs/similarity_matrix.html")
fig

In [15]:
fig = topic_model.visualize_hierarchy(color_threshold=0.9, custom_labels=True)
fig.write_html("../graphs/hierarchical_clustering.html")
fig

In [16]:
# Calculate topics over time
topics_over_time = topic_model.topics_over_time(docs, timestamp)
topics_over_time.to_csv('../bertopic/preprocessed_data/topics_over_time.csv')

0it [00:00, ?it/s]

1it [00:00,  3.71it/s]

2it [00:00,  3.27it/s]

3it [00:00,  3.19it/s]

4it [00:01,  3.15it/s]

5it [00:01,  3.25it/s]

... (39 outputs left out)

45it [00:53,  4.75s/it]

46it [00:58,  4.82s/it]

47it [00:58,  3.47s/it]

47it [00:58,  1.25s/it]

In [17]:
def visualize_topics_over_time(topic_model,
                               topics_over_time: pd.DataFrame,
                               top_n_topics: int = None,
                               topics: List[int] = None,
                               normalize_frequency: bool = False,
                               custom_labels: Union[bool, str] = False,
                               title: str = "<b>Topics over Time</b>",
                               width: int = 1250,
                               height: int = 450) -> go.Figure:
    """ Visualize topics over time
    Arguments:
        topic_model: A fitted BERTopic instance.
        topics_over_time: The topics you would like to be visualized with the
                          corresponding topic representation
        top_n_topics: To visualize the most frequent topics instead of all
        topics: Select which topics you would like to be visualized
        normalize_frequency: Whether to normalize each topic's frequency individually
        custom_labels: If bool, whether to use custom topic labels that were defined using 
                       `topic_model.set_topic_labels`.
                       If `str`, it uses labels from other aspects, e.g., "Aspect1".
        title: Title of the plot.
        width: The width of the figure.
        height: The height of the figure.
    Returns:
        A plotly.graph_objects.Figure including all traces
    Adapted from BERTopic's visualize_topics_over_time.
    """
    # Select topics based on top_n and topics args
    freq_df = topic_model.get_topic_freq()
    freq_df = freq_df.loc[freq_df.Topic != -1, :]
    if topics is not None:
        selected_topics = list(topics)
    elif top_n_topics is not None:
        selected_topics = sorted(freq_df.Topic.to_list()[:top_n_topics])
    else:
        selected_topics = sorted(freq_df.Topic.to_list())

    selected_topics.reverse()

    # Prepare data
    if isinstance(custom_labels, str):
        topic_names = [[[str(topic), None]] + topic_model.topic_aspects_[custom_labels][topic] for topic in topics]
        topic_names = ["_".join([label[0] for label in labels[:4]]) for labels in topic_names]
        topic_names = [label if len(label) < 30 else label[:27] + "..." for label in topic_names]
        topic_names = {key: topic_names[index] for index, key in enumerate(topic_model.topic_labels_.keys())}
    elif topic_model.custom_labels_ is not None and custom_labels:
        topic_names = {key: topic_model.custom_labels_[key + topic_model._outliers] for key, _ in topic_model.topic_labels_.items()}
    else:
        topic_names = {key: value[:40] + "..." if len(value) > 40 else value
                       for key, value in topic_model.topic_labels_.items()}
    topics_over_time["Name"] = topics_over_time.Topic.map(topic_names)
    data = topics_over_time.loc[topics_over_time.Topic.isin(selected_topics), :].sort_values(["Topic", "Timestamp"])
    
    # Add traces
    fig = go.Figure()
    for index, topic in enumerate(data.Topic.unique()):
        trace_data = data.loc[data.Topic == topic, :]
        topic_name = trace_data.Name.values[0]
        words = trace_data.Words.values
        if normalize_frequency:
            y = normalize(trace_data.Frequency.values.reshape(1, -1))[0]
        else:
            y = trace_data.Frequency

        fig.add_trace(go.Scatter(x=trace_data.Timestamp, y=y,
                                 mode='lines',
                                 marker_color=COLORS[index % len(COLORS)],
                                 hoverinfo="text",
                                 name=topic_name,
                                 hovertext=[f'<b>Topic {topic_name}</b><br>Words: {word}' for word in words]))

    # Styling of the visualization
    fig.update_xaxes(showgrid=True, range=[2008, REFERENCE_YEAR])
    fig.update_yaxes(showgrid=True)
    fig.update_layout(
        yaxis_title="Normalized N" if normalize_frequency else "N",
        title={
            'text': f"{title}",
            'y': .95,
            'x': 0.40,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(
                size=22,
                color="Black")
        },
        template="simple_white",
        width=width,
        height=height,
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Rockwell"
        ),
        legend=dict(
            title="<b>Topic",
            font=dict(size=12)
        )
    )
    return fig


In [18]:
# Plot topics over time up to the reference year
topics_over_time = pd.read_csv('../bertopic/preprocessed_data/topics_over_time.csv')
topics_over_time = topics_over_time[topics_over_time['Timestamp'] <= REFERENCE_YEAR]
fig = visualize_topics_over_time(topic_model, topics_over_time = topics_over_time, custom_labels=True, normalize_frequency=False, width=1250, height=900)
fig.write_html("../graphs/selected_topics_over_time.html")
fig

In [19]:
# Get papers per topic
docs_topics = pd.DataFrame({"Abstract": docs, "Topic": topic_model.topics_})
docs_topics['Topic names'] = docs_topics['Topic'].map(topic_labels)

# Save docs per topic and freq
docs_topics.to_csv("../bertopic/preprocessed_data/docs_topics.csv")
freq = topic_model.get_topic_info()
freq.to_csv("../bertopic/preprocessed_data/freq.csv")

Further explorations

In [20]:
docs_topics = pd.read_csv("../bertopic/preprocessed_data/docs_topics.csv")
freq = pd.read_csv("../bertopic/preprocessed_data/freq.csv")

# Create dataframe from cite, docs_topics, classes, id and timestamp
docs_topics_cite_year = pd.concat([ids, timestamp, cite, docs_topics, classes], axis=1)
docs_topics_cite_year.sort_values(by="Citation Count (Corrected by Age)", ascending=False, inplace=True)
docs_topics_cite_year.drop('Abstract', axis=1, inplace=True)
docs_topics_cite_year['Topic names'] = docs_topics_cite_year['Topic names'].fillna('Outliers')

# Visualize most cited papers
head = docs_topics_cite_year.head(20)
head

,Title,DOI,Authors,Year,Citation Count,Citation Count (Corrected by Age),Unnamed: 0,Topic,Topic names,Basin
11745,Plastic waste inputs from land into the ocean,10.1126/science.1260352,Jambeck J.R.; Geyer R.; Wilcox C.; Siegler T.R...,2015,7721,857.888889,11745,28,28: Global Ocean Plastic Waste Inputs and Clim...,Unknown
12138,Microplastics in the marine environment,10.1016/j.marpolbul.2011.05.030,Andrady A.L.,2011,5016,385.846154,12138,8,8: Microplastic Abundance and Distribution in ...,Unknown
10115,Emerging threats and persistent conservation c...,10.1111/brv.12480,Reid A.J.; Carlson A.K.; Creed I.F.; Eliason E...,2019,1802,360.400000,10115,-1,-1: Outliers,Equatorial Atlant
8612,Predicted growth in plastic waste exceeds effo...,10.1126/SCIENCE.ABA3656,Borrelle S.B.; Ringma J.; Lavender Law K.; Mon...,2020,1387,346.750000,8612,28,28: Global Ocean Plastic Waste Inputs and Clim...,Unknown
11167,River plastic emissions to the world's oceans,10.1038/ncomms15611,Lebreton L.C.M.; Van Der Zwet J.; Damsteeg J.-...,2017,2362,337.428571,11167,-1,-1: Outliers,North Indian
7224,The global threat from plastic pollution,10.1126/science.abg5433,MacLeod M.; Arp H.P.H.; Tekman M.B.; Jahnke A.,2021,989,329.666667,7224,2,"2: Plastic Waste Management, Governance and Mi...",Unknown
11771,Plastic Pollution in the World's Oceans: More ...,10.1371/journal.pone.0111913,Eriksen M.; Lebreton L.C.M.; Carson H.S.; Thie...,2014,3198,319.800000,11771,6,6: Modeling the Dispersal of Floating Plastic ...,South Pacific
11165,Microplastics in freshwater and terrestrial en...,10.1016/j.scitotenv.2017.01.190,Horton A.A.; Walton A.; Spurgeon D.J.; Lahive ...,2017,2229,318.428571,11165,36,36: Reviews of Micro- and Nanoplastic Fate and...,Unknown
9289,Environmental exposure to microplastics: An ov...,10.1016/j.scitotenv.2019.134455,Prata J.C.; da Costa J.P.; Lopes I.; Duarte A....,2020,1225,306.250000,9289,13,13: Human Exposure to Micro- and Nanoplastics ...,Unknown
12156,Microplastics as contaminants in the marine en...,10.1016/j.marpolbul.2011.09.025,Cole M.; Lindeque P.; Halsband C.; Galloway T.S.,2011,3931,302.384615,12156,-1,-1: Outliers,Unknown


In [21]:
# Topic frequency table: size, full name and mean age-corrected citations of each topic
topic_frequency = freq.loc[freq['Topic'] != -1, ['Topic', 'Count']].copy()
topic_frequency['Topics'] = topic_frequency['Topic'].map(topic_labels)
mean_citations = docs_topics_cite_year.groupby('Topic')['Citation Count (Corrected by Age)'].mean()
topic_frequency['Average citation'] = topic_frequency['Topic'].map(mean_citations)
topic_frequency.to_csv('../bertopic/preprocessed_data/topic_frequency.csv')
topic_frequency

,Topic,Count,Topics,Average citation
1,0,1500,0: Microplastic Abundance in Freshwater and Es...,12.392221
2,1,861,1: Toxic Effects of Micro- and Nanoplastic Exp...,13.132875
3,2,771,"2: Plastic Waste Management, Governance and Mi...",6.105583
4,3,728,3: Microplastic Contamination in Tissues of Co...,12.637534
5,4,683,4: Microbial Communities on Plastics (the Plas...,11.976019
6,5,676,5: Beach Litter Surveys and Marine Litter Moni...,5.811064
7,6,363,6: Modeling the Dispersal of Floating Plastic ...,15.125049
8,7,322,7: Microplastics in Soils and Terrestrial Ecos...,15.940292
9,8,318,8: Microplastic Abundance and Distribution in ...,15.500162
10,9,277,"9: Plastic Ingestion by Marine Fauna: Fish, Se...",9.276008


In [22]:
filtered_df = docs_topics_cite_year[(docs_topics_cite_year['Basin'] != 'Unknown') & (docs_topics_cite_year['Year'] <= REFERENCE_YEAR)]
basin_frequency = filtered_df.groupby(['Year', 'Basin']).size().unstack().fillna(0)
fig = px.line(basin_frequency,
              x=basin_frequency.index,
              y=basin_frequency.columns,
              color_discrete_sequence=COLORS,
              title='Frequency of Documents by Basin Over the Years (Excluding Unknown Basin)')
              
# Styling of the visualization
fig.update_xaxes(showgrid=True, range=[2008, REFERENCE_YEAR])
fig.update_yaxes(showgrid=True)
fig.update_layout(
        yaxis_title="N",
        title={
            'text': "Basins over year",
            'y': .95,
            'x': 0.40,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(
                size=22,
                color="Black")
        },
        template="simple_white",
        width=1250,
        height=900,
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Rockwell"
        ),
        legend=dict(
            title="<b>Basin",
        )
    )
fig


Heatmaps

In [23]:
# Plotly colorscales from matplotlib colormaps, with the lowest value (0 papers) shown in white
def white_start_colorscale(base_cmap):
    lower = np.array([[1.0, 1.0, 1.0, 0.0]])  # white, fully transparent
    cmap = mpl.colors.ListedColormap(np.vstack((lower, base_cmap(np.arange(256)))))
    return [mpl.colors.rgb2hex(color[:3]) for color in cmap(np.arange(cmap.N))]

plotly_colorscale = white_start_colorscale(cm.inferno)
plotly_colorscale2 = white_start_colorscale(cm.viridis)

In [24]:
# Group the dataframe by 'Topic' and 'Basin' and count the occurrences
grouped_df = docs_topics_cite_year.groupby(['Topic', 'Basin']).size().reset_index(name='Occurrences')

# Pivot the dataframe to get the occurrence counts as a matrix
pivot_df = grouped_df.pivot(index='Topic', columns='Basin', values='Occurrences').fillna(0)

# Basin and Topic size normalization
basin_counts = docs_topics_cite_year['Basin'].value_counts()
topic_counts = docs_topics_cite_year['Topic'].value_counts()

# Normalize the data by dividing each cell by the class size
pivot_scaled = pivot_df

# Sort the basins based on total occurrences in descending order
sorted_basins = basin_counts.sort_values(ascending=False).index

# Update the order of the basins in the pivot_scaled DataFrame
pivot_scaled_sorted = pivot_scaled[sorted_basins]

# Get the list of Topics in the same order as in the sorted pivot_scaled DataFrame
topics_ordered = pivot_scaled_sorted.index.tolist()

# Create a list of Topic Names corresponding to the Topics in the same order
topic_names_ordered = [topic_labels[topic] for topic in topics_ordered]

# Get the list of Basins (columns) in the same order as in the sorted pivot_scaled DataFrame
basins_ordered = pivot_scaled_sorted.columns.tolist()

# Create a 2D array of Topic Names for the hovertext
hovertext = [[f"Topic: {topic_names_ordered[i]}<br>Basin: {basins_ordered[j]}<br>Papers: {pivot_scaled_sorted.iloc[i, j]:.2f}"
              for j in range(len(basins_ordered))]
             for i, topic in enumerate(topics_ordered)]

# Create the Heatmap trace
heatmap_trace = go.Heatmap(z=pivot_scaled_sorted.values,
                           x=basins_ordered,
                           y=topics_ordered,
                           hovertext=hovertext,  # Use the hovertext created above
                           colorscale=plotly_colorscale,
                           hovertemplate='%{hovertext}<extra></extra>')  # Set the hovertemplate to only show the custom hovertext

# Create the figure and add the trace
fig = go.Figure(data=heatmap_trace)

# Customize the layout
fig.update_layout(
    font_size=15,
    title={
        'text': 'Topic vs Class Heatmap',
        'x': 0.5,  # Center the title
        'xanchor': 'center'
    },
    xaxis=dict(title='Basin'),
    yaxis=dict(title='Topic',
               dtick=2,
               autorange='reversed'),
    width=900,
    height=700,
)

# Add the colorbar and customize it
fig.update_coloraxes(
    colorbar=dict(
    yanchor='middle',  # Raise the colorbar title
    title='Papers',
    len=1,  # Adjust the length of the colorbar
    tickmode='array',
    tickfont=dict(color='black'),
    orientation='v',
))

# Show the plot
fig.write_html('../graphs/heatmap.html')
fig.show()

In [25]:
# Group the dataframe by 'Topic' and 'Basin' and count the occurrences
grouped_df = docs_topics_cite_year.groupby(['Topic', 'Basin']).size().reset_index(name='Occurrences')

# Pivot the dataframe to get the occurrence counts as a matrix
pivot_df = grouped_df.pivot(index='Topic', columns='Basin', values='Occurrences').fillna(0)

# Basin and Topic size normalization
basin_counts = docs_topics_cite_year['Basin'].value_counts()
topic_counts = docs_topics_cite_year['Topic'].value_counts()

# Sort the basins based on total occurrences in descending order
sorted_basins = basin_counts.sort_values(ascending=False).index

# Update the order of the basins in the pivot_scaled DataFrame
pivot_sorted = pivot_df[sorted_basins]

# apply log to the data
pivot_scaled_sorted = np.log1p(pivot_sorted)

# Get the list of Topics in the same order as in the sorted pivot_scaled DataFrame
topics_ordered = pivot_scaled_sorted.index.tolist()

# Create a list of Topic Names corresponding to the Topics in the same order
topic_names_ordered = [topic_labels[topic] for topic in topics_ordered]

# Get the list of Basins (columns) in the same order as in the sorted pivot_scaled DataFrame
basins_ordered = pivot_scaled_sorted.columns.tolist()

# Create a 2D array of Topic Names for the hovertext
hovertext = [[f"Topic: {topic_names_ordered[i]}<br>Basin: {basins_ordered[j]}<br>Log N: {pivot_scaled_sorted.iloc[i, j]:.2f}<br> N: {pivot_sorted.iloc[i, j]:.2f}"
              for j in range(len(basins_ordered))]
             for i, topic in enumerate(topics_ordered)]

# Create the Heatmap trace
heatmap_trace = go.Heatmap(
    z=pivot_scaled_sorted.values,
    x=basins_ordered,
    y=topics_ordered,
    hovertext=hovertext,
    colorscale=plotly_colorscale,
    hovertemplate='%{hovertext}<extra></extra>',
    colorbar=dict(
        yanchor='middle',  # Raise the colorbar title
        title='Log N',
        tickmode='array',
        tickfont=dict(color='black'),
        tickvals=[0, 1, 2, 3, 4, 5],  # Adjust tickvals as needed
        ticktext=['0', '1', '2', '3', '4', '5'],  # Adjust ticktext as needed
    )
)

# Create the figure and add the trace
fig = go.Figure(data=heatmap_trace)

# Customize the layout
fig.update_layout(
    font_size=15,
    title={
        'text': 'Topic vs Class Heatmap',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis=dict(title='Basin'),
    yaxis=dict(title='Topic', dtick=2, autorange='reversed'),
    width=900,
    height=700,
)

# Show the plot
fig.write_html('../graphs/log_heatmap.html')
fig.show()

In [26]:
# Group the dataframe by 'Topic' and 'Basin' and count the occurrences
avg_citation = docs_topics_cite_year['Citation Count'].mean()

# Assuming your dataframe is named 'df'
grouped_df = docs_topics_cite_year.groupby(['Basin', 'Topic'])['Citation Count'].mean().reset_index()

# Pivot the dataframe to get the occurrence counts as a matrix
pivot_df = grouped_df.pivot(index=['Topic'], columns='Basin', values='Citation Count').fillna(0)

# Basin and Topic size normalization
basin_counts = docs_topics_cite_year['Basin'].value_counts()

# Normalize the data by dividing each cell by the class size
pivot_scaled = np.log1p(pivot_df)

# Sort the basins based on total occurrences in descending order
sorted_basins = basin_counts.sort_values(ascending=False).index

# Update the order of the basins in the pivot_scaled DataFrame
pivot_scaled_sorted = pivot_scaled[sorted_basins]

# Get the list of Topics in the same order as in the sorted pivot_scaled DataFrame
topics_ordered = pivot_scaled_sorted.index.tolist()

# Create a list of Topic Names corresponding to the Topics in the same order
topic_names_ordered = [topic_labels[topic] for topic in topics_ordered]

# Get the list of Basins (columns) in the same order as in the sorted pivot_scaled DataFrame
basins_ordered = pivot_scaled_sorted.columns.tolist()

# Create a 2D array of Topic Names for the hovertext
hovertext = [[f"Topic: {topic_names_ordered[i]}<br>Basin: {basins_ordered[j]}<br>Average Citations: {pivot_scaled_sorted.iloc[i, j]:.2f}"
              for j in range(len(basins_ordered))]
             for i, topic in enumerate(topics_ordered)]

# Create the Heatmap trace
heatmap_trace = go.Heatmap(z=pivot_scaled_sorted.values,
                           x=basins_ordered,
                           y=topics_ordered,
                           hovertext=hovertext,  # Use the hovertext created above
                           colorscale=plotly_colorscale2,           
                           hovertemplate='%{hovertext}<extra></extra>')  # Set the hovertemplate to only show the custom hovertext

# Create the figure and add the trace
fig = go.Figure(data=heatmap_trace)

# Customize the layout
fig.update_layout(    
    font_size=15,
    title={
        'text': 'Topic vs Class Heatmap',
        'x': 0.5,  # Center the title
        'xanchor': 'center'
    },
    xaxis=dict(title='Basin', tickangle=30),
    yaxis=dict(title='Topic',
               dtick=2,
               autorange='reversed'),
    width=900,
    height=700,
)

# Add the colorbar and customize it
fig.update_coloraxes(
    colorbar=dict(
    yanchor='middle',  # Raise the colorbar title
    title='Log Mean<br>Citations',
    len=1,  # Adjust the length of the colorbar
    tickmode='array',
    tickfont=dict(color='black'),
    orientation='v',
))

# Display the heatmap
fig.show()

Gini

In [27]:
# Gini coefficient of the citations (0 = equally cited, 1 = all citations in one paper)
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n == 0:
        return 0
    # Equivalent to sum(|xi - xj|) / (2 n^2 mean(x)), computed on the sorted values
    return np.sum((2 * np.arange(1, n + 1) - n - 1) * x) / (n * x.sum())

# Calculate gini coefficient for the whole dataframe
gini_coefficient = gini(docs_topics_cite_year["Citation Count (Corrected by Age)"])
print("Gini Coefficient for the entire DataFrame:", gini_coefficient)

Gini Coefficient for the entire DataFrame: 0.6605144203027183


In [28]:
# Group by 'Topic' and calculate the Gini coefficients
giniall = docs_topics_cite_year.groupby(["Topic"])["Citation Count (Corrected by Age)"].apply(gini)
giniall = giniall.reset_index(level=['Topic'])
giniall = giniall.rename(columns={"Citation Count (Corrected by Age)": 'Gini Coefficient'})
giniall

,Topic,Gini Coefficient
0,-1,0.670944
1,0,0.604294
2,1,0.586265
3,2,0.719662
4,3,0.601658
5,4,0.652029
6,5,0.560417
7,6,0.702974
8,7,0.690298
9,8,0.665560


In [29]:
# Calculcate average of topics' gini coefficient
print(giniall['Gini Coefficient'].mean())

0.63425616938898


In [30]:
# Group by 'Topic' and 'Year' and calculate the Gini coefficients for each group
gini_topic_year = docs_topics_cite_year.groupby(["Topic", "Year"])["Citation Count (Corrected by Age)"].apply(gini)
gini_topic_year = gini_topic_year.reset_index(level=['Topic','Year'])
gini_topic_year = gini_topic_year.rename(columns={"Citation Count (Corrected by Age)": 'Gini Coefficient'})
gini_topic_year.to_csv("../bertopic/preprocessed_data/gini.csv")
gini_topic_year


/var/folders/wd/2xd6gf696k90tw1t6pxntp4r0000gn/T/ipykernel_4220/3589637708.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum((2 * np.arange(1, n + 1) - n - 1) * x) / (n * x.sum())


,Topic,Year,Gini Coefficient
0,-1,1981,0.358824
1,-1,1987,NaN
2,-1,1990,0.000000
3,-1,1992,0.000000
4,-1,1993,0.488636
...,...,...,...
575,38,2020,0.484463
576,38,2021,0.353875
577,38,2022,0.375887
578,38,2023,0.468905


In [31]:
gini_topic_year = pd.read_csv("../bertopic/preprocessed_data/gini.csv")

# Group by 'Topic' and 'Year' and calculate the sum of publications for each group
num_publications = docs_topics_cite_year.groupby(['Topic', 'Year']).size().reset_index(name='Occurrences')

# Merge the two DataFrames on "Topic" and "Year"
merged_df = pd.merge(num_publications, gini_topic_year, on=["Topic", "Year"])

# Extract the number of publications as a list to use as x-axis
num_publications_values = np.log10(merged_df['Occurrences'])
merged_df['Topic'] = merged_df['Topic'].astype(str)

# Create a scatter plot with color-coded topics
fig = px.scatter(merged_df, x=num_publications_values, y='Gini Coefficient', color='Topic',
                 labels={'x': 'Log N', 'y': 'Gini Coefficient (Citations/year)'},
                 color_discrete_sequence=COLORS,
                 opacity=0.5,
                 trendline='lowess')

fig.update_layout(width=900, height=700, font_size=15,
                  template="simple_white")
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.write_html('../graphs/gini_logN.html')
fig.show()

In [32]:
# Group by 'Topic' and 'Year' and calculate the sum of publications for each group
num_publications = docs_topics_cite_year.groupby(['Topic', 'Year']).size().reset_index(name='Occurrences')

# Merge the two DataFrames on "Topic" and "Year"
merged_df = pd.merge(num_publications, gini_topic_year, on=["Topic", "Year"])

# Filter the DataFrame to include only the selected topics
df_filtered = merged_df[merged_df['Topic'].isin(SELECTED_TOPICS)].copy()

# Extract the number of publications as a list to use as x-axis
num_publications_values = np.log10(df_filtered['Occurrences'])

df_filtered['Topic'] = df_filtered['Topic'].astype(str)

# Create a scatter plot with color-coded topics
fig = px.scatter(df_filtered, x=num_publications_values, y='Gini Coefficient', color='Topic',
                 labels={'x': 'Log N', 'y': 'Gini Coefficient (Citations/year)'},
                 color_discrete_sequence=COLORS,
                 opacity=0.5,
                 trendline='lowess')

fig.update_layout(width=900, height=700,
                  template="simple_white", font_size=15)
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.write_html('../graphs/selected_gini_logN.html')
fig.show()

Years with more publications per topic have a higher gini coefficient. This means that each year with large number of publications, has less highly citated papers. Which means that citations are more concentrated in few papers in years with more publications

In [33]:
# Gini coefficient per topic and year, up to the reference year
merged_df['Topic'] = merged_df['Topic'].astype(str)
merged_df = merged_df[merged_df['Year'] <= REFERENCE_YEAR]

# Create a scatter plot with color-coded topics
fig = px.scatter(merged_df, x='Year', y='Gini Coefficient', color='Topic',
                 labels={'x': 'Log N', 'y': 'Gini Coefficient (Citations/year)'},
                 color_discrete_sequence=COLORS,
                 opacity=0.5,
                 trendline='lowess')

fig.update_layout(width=900, height=700,
                  template="simple_white", font_size=15)
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.write_html('../graphs/gini_year.html')
fig.show()

For all papers published in a given year, the amount of citations given to just a few papers increases over the years.
For instace, on one year, a topic had 3 publications. we tend to cite them more equally than on a year with 100 publications.

In [34]:
# Sort the papers of each topic by citation count, in ascending order
df_sorted = docs_topics_cite_year.sort_values(['Topic', 'Citation Count (Corrected by Age)']).reset_index(drop=True)

# Calculate the cumulative count of papers for each topic starting from zero
df_sorted['cumulative_paper_count'] = df_sorted.groupby('Topic').cumcount() + 1

# Calculate the cumulative sum of citations for each topic group
df_sorted['cumulative_sum'] = df_sorted.groupby('Topic')['Citation Count (Corrected by Age)'].cumsum()

# Find the total number of citations in each topic group
topic_citations_total = df_sorted.groupby('Topic')['Citation Count (Corrected by Age)'].sum()

# Merge with topic_citations_total to get identical index
df_sorted = df_sorted.merge(topic_citations_total.rename('topic_citations_total'), on='Topic')

# Create a new column for the normalized cumulative paper count
df_sorted['normalized_cumulative_paper_count'] = df_sorted.groupby('Topic')['cumulative_paper_count'].transform(lambda x: x / x.max())

# Create a new column for the normalized cumulative sum
df_sorted['normalized_cumulative_sum'] = df_sorted.groupby('Topic')['cumulative_sum'].transform(lambda x: x / x.max())

# Display the resulting DataFrame
df_sorted

,Title,DOI,Authors,Year,Citation Count,Citation Count (Corrected by Age),Unnamed: 0,Topic,Topic names,Basin,cumulative_paper_count,cumulative_sum,topic_citations_total,normalized_cumulative_paper_count,normalized_cumulative_sum
0,"Advances in the role of microorganisms, waste ...",10.1016/j.scp.2024.101595,Rani R.; Malik S.; Kumar D.; Kumar R.; Mukherj...,2024,0,0.000000,491,-1,-1: Outliers,Unknown,1,0.000000,31360.099974,0.000369,0.000000
1,"Microplastic pollution in Vietnam's estuarine,...",10.1016/j.ecss.2024.108749,Veettil B.K.; Puri V.; Acharki S.; Ward R.D.; ...,2024,0,0.000000,494,-1,-1: Outliers,North Pacific,2,0.000000,31360.099974,0.000738,0.000000
2,"Profile, Tissue Distribution, and Time Trend o...",10.1002/etc.5715,Peng X.; Zhou J.; Chen G.; Tan J.; Zhu Z.,2023,0,0.000000,2194,-1,-1: Outliers,North Pacific,3,0.000000,31360.099974,0.001107,0.000000
3,Aging increases the particulate- and leachate-...,10.1016/j.watres.2024.121653,Lv M.; Meng F.; Man M.; Lu S.; Ren S.; Yang X....,2024,0,0.000000,486,-1,-1: Outliers,Unknown,4,0.000000,31360.099974,0.001477,0.000000
4,Evaluating community science sampling for micr...,10.1139/facets-2022-0104,Forrest S.A.; Alambo K.; Holman L.; Vermaire J.C.,2022,0,0.000000,6084,-1,-1: Outliers,North Atlantic,5,0.000000,31360.099974,0.001846,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12393,Occurrences of organophosphorus esters and pht...,10.1016/j.scitotenv.2017.10.163,Zhang H.; Zhou Q.; Xie Z.; Zhou Y.; Tu C.; Fu ...,2018,151,25.166667,10970,38,38: PAHs and Other Organic Pollutants Associat...,North Pacific,44,293.989394,736.239394,0.916667,0.399312
12394,Assessment of cancer risk of microplastics enr...,10.1016/j.jhazmat.2020.122994,Sharma M.D.; Elanjickal A.I.; Mankar J.S.; Kru...,2020,141,35.250000,8332,38,38: PAHs and Other Organic Pollutants Associat...,Unknown,45,329.239394,736.239394,0.937500,0.447191
12395,Microplastic in Aquatic Ecosystems,10.1002/anie.201606957,Ivleva N.P.; Wiesheu A.C.; Niessner R.,2017,558,79.714286,11286,38,38: PAHs and Other Organic Pollutants Associat...,North Atlantic,46,408.953680,736.239394,0.958333,0.555463
12396,Toxicities of polycyclic aromatic hydrocarbons...,10.3390/ijerph17041363,Honda M.; Suzuki N.,2020,328,82.000000,9359,38,38: PAHs and Other Organic Pollutants Associat...,North Atlantic,47,490.953680,736.239394,0.979167,0.666840


In [35]:
# Filter the DataFrame to include only the selected topics
df_filtered = df_sorted[df_sorted['Topic'].isin(SELECTED_TOPICS)].copy()
df_filtered['Topic'] = df_filtered['Topic'].astype(str)

# Create traces for each topic in the main curve
traces_main_curve = []
for topic, color in zip(SELECTED_TOPICS, COLORS):
    df_topic = df_filtered[df_filtered['Topic'] == str(topic)]
    trace_lines = go.Scatter(
        x=df_topic['normalized_cumulative_paper_count'],
        y=df_topic['normalized_cumulative_sum'],
        mode='lines',
        line=dict(color=color),
        showlegend=False
    )
    trace_markers = go.Scatter(
        x=df_topic['normalized_cumulative_paper_count'],
        y=df_topic['normalized_cumulative_sum'],
        mode='markers',
        marker=dict(size=5, opacity=0.7, color=color),
        name=topic
    )
    traces_main_curve.extend([trace_lines, trace_markers])



# Add a straight line from (0, 0) to (1, 1) representing perfect equality in citation distribution
trace_equality_line = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(color='black', dash='dash'),
    name='Equality Line'
)

# Add a straight line representing high impact publications
trace_high_impact_line = go.Scatter(
    x=[0.9, 0.9],
    y=[0, 1],
    mode='lines',
    line=dict(color='red', dash='solid'),
    name='High Impact'
)

# Create the layout
layout = go.Layout(
    width=900,
    height=700,
    font_size=15,
    xaxis=dict(title='Cumulative share of publications'),
    yaxis=dict(title='Cumulative share of citations'),
    template="simple_white",
    showlegend=True
)

# Create the figure
fig = go.Figure(data=traces_main_curve + [trace_equality_line, trace_high_impact_line], layout=layout)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.write_html('../graphs/cumulative_citations_selected.html')
fig.show()


Topics need between 86 and 100 % to get half of their citations. between 14 and 0 percent of all papers have 50% of the citations